In [ ]:
import os, sys
cur_directory = os.getcwd()
project_folder = os.path.abspath(os.path.join(cur_directory, os.pardir, os.pardir))
sys.path.append(os.path.join(project_folder, 'analysis-suite'))

In [ ]:
import numpy as np
from numpy import linalg as la
import matplotlib as mpl
from matplotlib import pyplot as plt
import analyses as als
import datasets as ds
import models as mds
import tools
from random import sample
from tqdm import tqdm
from argparse import Namespace
from time import gmtime, strftime
import pickle

## Params

In [ ]:
V_width = 1
V_E_buffer_array = np.arange(20)
E_width = 1

# neuron_inds = range(53)
neuron_inds = range(93)

N_training_repeats = 400
K = 10
# K_array = np.arange(4).astype(int) + 1 # number of meta neurons to be included

cg_params = {
    "order": 1,
}

critic_params = {
    "hidden_dim": 32,
    "embed_dim": 8,
    "layers": 2,
    "activation": "tanh",
}
opt_params = {"track_quantities": False, "quiet": True, "iterations": 1}

train_fraction = 0.5
samp_keep = 1

# Put variable names here for parameters we want saved.
save_vars = {
    "N_training_repeats",
    "neuron_inds",
    "cg_params",
    "critic_params",
    "opt_params",
    "V_width",
    "V_E_buffer_array",
    "E_width",
    "train_fraction",
    "samp_keep",
    "K"
}

In [ ]:
V_E_buffer_array

## Prepare data (Nat Scenes)

In [ ]:
data_path = os.path.join(project_folder, 'data', 'nat_scenes_data')
D = ds.DataNatScenes(data_path, mov_ind=4)
print(f'current stimulus is {D.movname}')
mov_ind = D.mov_ind
mov_name = D.movname
stim = D.movname
save_vars.add('mov_ind')
save_vars.add('mov_name')

## Prepare data (bar)

In [ ]:
# data_path = os.path.join(project_folder, 'data', 'bar_stim_data')
# D = ds.DataBar(data_path)
# stim = "bar"
# mov_name = "bar"
# save_vars.add("mov_name")

## Run analysis

In [ ]:
experiment_dims = (len(V_E_buffer_array), N_training_repeats,)

raw_mn_array = np.zeros(experiment_dims, dtype=object)
critic_function_array = np.zeros(experiment_dims, dtype=object)
train_points = np.zeros(experiment_dims, dtype=object)
test_points = np.zeros(experiment_dims, dtype=object)

 
iterated_save_vars = {
        "raw_mn_array",
        "train_points",
    }

for i, V_E_buffer in tqdm(enumerate(V_E_buffer_array)):
    for training_repeat in range(N_training_repeats):
        trial_inds = (i, training_repeat,)

        # Domains, training and test subsets
        D.reset()
        D.trim(20)
        # D.subset_features(sort_by_cov_inds)  # sort neurons by VE autocorrelation.
        D.subset_features(neuron_inds)
        V, E = D.create_V_and_E_datasets(V_width, V_E_buffer, E_width, samp_keep)
        (current_train_points, current_test_points,) = tools.generate_train_test_indices(V.P, train_fraction)
        train_points[trial_inds] = current_train_points
        test_points[trial_inds] = current_test_points

        # initialize Ansaetze
        coarse_grainer = mds.PolynomialLayer(output_dim=K, **cg_params)
        critic_function = mds.SeparableCritic(**critic_params)

        # Train current
        V.subset_points(current_train_points)
        E.subset_points(current_train_points)
        als.IB(
            X=V.X,
            Y=E.X,
            coarse_grainer=coarse_grainer,
            critic_function=critic_function,
            **opt_params)
        
        critic_function_array[trial_inds] = critic_function
        raw_mn_array[trial_inds] = coarse_grainer.get_weights()[1]



## Process solutions and get basis meta neurons

In [ ]:
meta_neurons = np.zeros(len(V_E_buffer_array), dtype=object)
basis_vectors = np.zeros(len(V_E_buffer_array), dtype=object)
lam = np.zeros(len(V_E_buffer_array), dtype=object)

for i, V_E_buffer in enumerate(V_E_buffer_array):

    mn_weights_combined = np.concatenate(raw_mn_array[i], axis=1)
    P = K * N_training_repeats

    # normalize filters
    A = mn_weights_combined / np.sqrt(np.sum(mn_weights_combined**2, axis=0, keepdims=True))

    # align and sort using PCA
    lam[i], basis_vectors[i] = als.pca(A)
    q = basis_vectors[i].T @ A
    cluster_id = np.argmax(np.abs(q), axis=0)
    relevant_sign = np.sign(q[cluster_id,np.arange(P)])
    A = A * relevant_sign

    meta_neurons[i] = A

iterated_save_vars.update({
    "meta_neurons",
    "basis_vectors",
    "lam",
})

## Save

In [ ]:
save_directory = os.path.join(cur_directory, 'saved', stim)
print(f"will save to {save_directory}")

In [ ]:
# select variables

critic_type = type(critic_function)
coarse_grainer_type = type(coarse_grainer)
save_vars.update({"critic_type", "coarse_grainer_type"})
local_dict = locals()

# for every step in number of meta neurons, save a different set of data
for i, V_E_buffer in tqdm(enumerate(V_E_buffer_array)):
    output_data = {k: local_dict[k] for k in save_vars}
    output_data.update({k: local_dict[k][i] for k in iterated_save_vars})
    full_file_path = os.path.join(save_directory, f'({V_width},{V_E_buffer},{E_width}) K={K}.dat')
    os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
    with open(full_file_path, 'wb+') as f:
            pickle.dump(output_data, f)
    tools.record_parameters(full_file_path, output_data)

## Reload

In [ ]:
load_path = project_folder + f'analyses/basis meta neuron bank/saved/{stim}/(1,1,1) K=10.dat'
with open(load_path, 'rb') as f:
    data_dict = pickle.load(f)
data = Namespace(**data_dict)

In [ ]:
data_dict.keys()